<a href="https://colab.research.google.com/github/Naimish-Raj/Assignment_01/blob/main/Copy_of_FinalYear_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: INSTALLATION
!pip install -q -U \
    transformers \
    sentence-transformers \
    tavily-python \
    python-dotenv \
    accelerate

    # INSTALL BITSANDBYTES
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.8 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 2: IMPORTS + GPU CHECK + TAVILY SETUP
# Google Colab Compatible
# ============================================================

import os
import torch
from getpass import getpass

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)

from sentence_transformers import SentenceTransformer
from tavily import TavilyClient

# ============================================================
# GPU STATUS
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("DEVICE STATUS")
print("=" * 60)
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("WARNING: GPU not available (Running on CPU).")

print("=" * 60)

# ============================================================
# TAVILY API KEY
# ============================================================

TAVILY_API_KEY = getpass("Enter your Tavily API Key: ").strip()

tavily_client = None
TAVILY_AVAILABLE = False

if len(TAVILY_API_KEY) < 10:
    print("\nERROR: Invalid or empty Tavily API key.")

else:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

    try:
        tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
        print("\nTavily client initialized.")
    except Exception as e:
        print("\nFailed to initialize Tavily client.")
        print("Error:", str(e))

# ============================================================
# TEST TAVILY CONNECTION
# ============================================================

if tavily_client is not None:

    print("\nTesting Tavily API...")

    try:
        response = tavily_client.search(
            query="Who is the current Governor of Bihar?",
            search_depth="basic",
            max_results=2
        )

        results = response.get("results", [])

        if results:
            TAVILY_AVAILABLE = True
            print("Tavily API: SUCCESS")
            print("Results found:", len(results))
        else:
            print("Tavily API connected, but no results returned.")

    except Exception as e:
        print("Tavily API: FAILED")
        print("Reason:", str(e))
        print("Continuing without web search...")
        TAVILY_AVAILABLE = False

# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 60)
print("SYSTEM STATUS")
print("=" * 60)
print(f"GPU Ready: {torch.cuda.is_available()}")
print(f"Tavily Ready: {TAVILY_AVAILABLE}")
print("=" * 60)

DEVICE STATUS
Device: cuda
GPU: Tesla T4
CUDA Version: 12.8
Enter your Tavily API Key: ··········

Tavily client initialized.

Testing Tavily API...
Tavily API: SUCCESS
Results found: 2

SYSTEM STATUS
GPU Ready: True
Tavily Ready: True


In [ ]:
# ============================================================
# DIAGNOSTIC CELL
# Run this BEFORE Cell 3
# ============================================================

import sys
import torch
import transformers

print("=" * 70)
print("PYTHON / PYTORCH / CUDA / TRANSFORMERS CHECK")
print("=" * 70)

# Python
print(f"Python Version: {sys.version.split()[0]}")

# PyTorch
print(f"PyTorch Version: {torch.__version__}")

# CUDA
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
print(f"CUDA Version: {torch.version.cuda}")

if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / (1024**3):.2f} GB")
    print(f"Compute Capability: {props.major}.{props.minor}")
else:
    print("Running on CPU.")

# Transformers
print(f"Transformers Version: {transformers.__version__}")

# ------------------------------------------------------------
# bitsandbytes Check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BITSANDBYTES CHECK")
print("=" * 70)

bnb_available = False

try:
    import bitsandbytes as bnb

    bnb_available = True
    print("bitsandbytes imported successfully!")
    print(f"Version: {bnb.__version__}")

except Exception as e:
    print("bitsandbytes IMPORT FAILED!")
    print("Error:", str(e))

    if not cuda_available:
        print("Reason: CPU runtime detected. This is usually expected.")

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ENVIRONMENT SUMMARY")
print("=" * 70)
print(f"Python OK        : True")
print(f"PyTorch OK       : True")
print(f"CUDA Enabled     : {cuda_available}")
print(f"BitsAndBytes OK  : {bnb_available}")
print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

PYTHON / PYTORCH / CUDA / TRANSFORMERS CHECK
Python Version: 3.13.15
PyTorch Version: 2.11.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU: Tesla T4
GPU Memory: 14.56 GB
Compute Capability: 7.5
Transformers Version: 5.17.0

BITSANDBYTES CHECK
bitsandbytes imported successfully!
Version: 0.50.2

ENVIRONMENT SUMMARY
Python OK        : True
PyTorch OK       : True
CUDA Enabled     : True
BitsAndBytes OK  : True
DIAGNOSTIC COMPLETE


In [ ]:
# CELL 3: LOAD ALL AI MODELS
# Mistral-7B-Instruct-v0.3 + 4-bit Quantization
# Sentence Transformer + NLI

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)

from sentence_transformers import SentenceTransformer


# ============================================================
# CHECK GPU
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU available . "
        "Colab T4 GPU select ."
    )

print("=" * 60)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("=" * 60)


# ============================================================
# MODEL 1: MISTRAL 7B INSTRUCT
# ============================================================

LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

print("\n[1/3] Loading Mistral-7B-Instruct-v0.3...")


# ------------------------------------------------------------
# 4-BIT QUANTIZATION
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)


# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL
)


# ------------------------------------------------------------
# PAD TOKEN
# ------------------------------------------------------------

if llm_tokenizer.pad_token is None:

    llm_tokenizer.pad_token = (
        llm_tokenizer.eos_token
    )


# ------------------------------------------------------------
# LOAD MISTRAL
# ------------------------------------------------------------

llm_model = AutoModelForCausalLM.from_pretrained(

    LLM_MODEL,

    quantization_config=bnb_config,

    device_map="auto",

    torch_dtype=torch.float16
)


# Evaluation mode

llm_model.eval()


print(
    "Mistral-7B-Instruct-v0.3 loaded successfully!"
)


# ============================================================
# MODEL 2: SENTENCE TRANSFORMER
# ============================================================

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

print(
    "\n[2/3] Loading Sentence Transformer..."
)


embedding_model = SentenceTransformer(

    EMBEDDING_MODEL,

    device=str(device)
)


print(
    "Embedding model loaded!"
)


# ============================================================
# MODEL 3: NLI
# ============================================================

NLI_MODEL = (
    "MoritzLaurer/"
    "DeBERTa-v3-base-mnli-fever-anli"
)

print(
    "\n[3/3] Loading NLI model..."
)


nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL
)


nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL
)


nli_model.to(device)

nli_model.eval()


print(
    "NLI model loaded!"
)


# ============================================================
# NLI LABELS
# ============================================================

print("\nNLI Labels:")

print(
    nli_model.config.id2label
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 60)

print(
    "ALL MODELS LOADED SUCCESSFULLY!"
)

print(
    "Device:",
    device
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print("=" * 60)

GPU: Tesla T4
CUDA: 12.8

[1/3] Loading Mistral-7B-Instruct-v0.3...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI model loaded!

NLI Labels:
{0: 'entailment', 1: 'neutral', 2: 'contradiction'}

ALL MODELS LOADED SUCCESSFULLY!
Device: cuda
GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 4: CORE HALLUCINATION DETECTION FUNCTIONS
# MISTRAL + TAVILY + SEMANTIC SIMILARITY + NLI + RAG
# ============================================================

import re
import numpy as np
import gc
from datetime import datetime
from urllib.parse import urlparse


# ============================================================
# HELPER FUNCTION — SHARED MISTRAL GENERATION
# ============================================================

def _generate_with_mistral(messages, max_new_tokens, max_input_length=2048):
    """
    Shared helper for all Mistral generation calls.
    Handles tokenization, generation, decoding, and GPU memory cleanup.
    """

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = llm_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=llm_tokenizer.pad_token_id,
            eos_token_id=llm_tokenizer.eos_token_id
        )

    generated_tokens = output[0, inputs["input_ids"].shape[-1]:]
    answer = llm_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    del inputs, output, generated_tokens
    torch.cuda.empty_cache()
    gc.collect()

    return answer


# ============================================================
# TRUSTED / UNRELIABLE DOMAINS
# ============================================================

UNRELIABLE_DOMAINS = [
    "facebook.com",
    "twitter.com",
    "x.com",
    "instagram.com",
    "reddit.com",
    "quora.com",
    "pinterest.com",
    "tiktok.com",
    "tripadvisor.com",
    "tripadvisor.in"
]


TRUSTED_DOMAINS = [
    "wikipedia.org",
    "britannica.com",
    "reuters.com",
    "apnews.com",
    "bbc.com",
    "theguardian.com",
    "who.int",
    "un.org",
    "nature.com",
    "sciencedirect.com",
    "ncbi.nlm.nih.gov",
    "whc.unesco.org",
    "unesco.org"
]


# ============================================================
# HELPER: CHECK TRUSTED DOMAIN
# ============================================================

def is_trusted_url(url):

    if not url:
        return False

    try:
        hostname = urlparse(url).netloc.lower()
        hostname = hostname.replace("www.", "")

        for domain in TRUSTED_DOMAINS:
            domain = domain.lower()
            if hostname == domain or hostname.endswith("." + domain):
                return True

        return False

    except Exception:
        return False


# ============================================================
# HELPER: CHECK UNRELIABLE DOMAIN
# ============================================================

def is_unreliable_url(url):

    if not url:
        return True

    try:
        hostname = urlparse(url).netloc.lower()
        hostname = hostname.replace("www.", "")

        for domain in UNRELIABLE_DOMAINS:
            domain = domain.lower()
            if hostname == domain or hostname.endswith("." + domain):
                return True

        return False

    except Exception:
        return True


# ============================================================
# 1. MISTRAL ANSWER GENERATION (PARAMETRIC — NO EVIDENCE)
# ============================================================

def generate_llm_answer(question, max_new_tokens=150):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful and factual AI assistant. "
                "Answer the user's question clearly and directly. "
                "Do not repeat the question. "
                "Do not intentionally invent facts. "
                "If the answer may be outdated or uncertain, "
                "say that you are uncertain."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    return _generate_with_mistral(
        messages,
        max_new_tokens=max_new_tokens,
        max_input_length=2048
    )


# ============================================================
# 2. CLAIM EXTRACTION
# ============================================================

def extract_claims(answer):

    if not answer or not answer.strip():
        return []

    answer = re.sub(r"(?m)^\s*[-•*]\s*", "", answer)
    answer = re.sub(r"\s+", " ", answer).strip()

    sentences = re.split(r"(?<=[.!?])\s+", answer)

    claims = []
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) < 15:
            continue
        claims.append(sentence)

    if not claims and len(answer) >= 15:
        claims.append(answer)

    return claims


# ============================================================
# 3. TIME SENSITIVE CHECK
# ============================================================

def is_time_sensitive(claim):

    current_year = datetime.now().year

    recent_years = [
        str(current_year),
        str(current_year + 1),
        str(current_year - 1)
    ]

    if any(year in claim for year in recent_years):
        return True

    recency_keywords = [
        "current", "currently", "latest", "recent", "recently",
        "now", "today", "this year", "upcoming", "ongoing",
        "winner", "champion", "elected", "president", "prime minister"
    ]

    claim_lower = claim.lower()

    if any(keyword in claim_lower for keyword in recency_keywords):
        return True

    return False


# ============================================================
# 4. EXTERNAL EVIDENCE RETRIEVAL (WITH FALLBACK)
# ============================================================

def _process_tavily_results(raw_results, restrict_to_trusted):
    """
    Shared logic for turning raw Tavily results into a clean
    evidence list. If restrict_to_trusted is True, only URLs on
    TRUSTED_DOMAINS pass. Either way, known unreliable domains
    (social media, forums, etc.) are always excluded.
    """
    processed = []

    for item in raw_results:
        content = item.get("content", "")
        url = item.get("url", "")
        title = item.get("title", "")

        if not content:
            continue

        if is_unreliable_url(url):
            continue

        if restrict_to_trusted and not is_trusted_url(url):
            continue

        processed.append({"title": title, "content": content, "url": url})

    return processed


def retrieve_evidence(claim, max_results=5):

    if "tavily_client" not in globals() or tavily_client is None:
        print("Tavily client is not available.")
        return []

    time_filter = {}
    if is_time_sensitive(claim):
        time_filter["time_range"] = "year"
        print("[Time-sensitive claim detected]")

    try:
        # --------------------------------------------------------
        # Attempt 1: trusted domains only (highest quality sources)
        # --------------------------------------------------------
        print("Searching trusted domains...")

        response = tavily_client.search(
            query=claim,
            search_depth="advanced",
            max_results=max_results,
            include_answer=False,
            include_domains=TRUSTED_DOMAINS,
            **time_filter
        )

        processed = _process_tavily_results(
            response.get("results", []), restrict_to_trusted=True
        )

        # --------------------------------------------------------
        # Attempt 2: fallback to broader web search
        # (still excludes clearly unreliable domains like social media)
        # --------------------------------------------------------
        if not processed:
            print("No trusted-domain results. Falling back to broader search...")

            response = tavily_client.search(
                query=claim,
                search_depth="advanced",
                max_results=max_results,
                include_answer=False,
                **time_filter
            )

            processed = _process_tavily_results(
                response.get("results", []), restrict_to_trusted=False
            )

        print(f"Usable evidence: {len(processed)}")
        return processed[:max_results]

    except Exception as e:
        print("Tavily error:", e)
        return []


# ============================================================
# 5. SEMANTIC SIMILARITY
# ============================================================

def semantic_similarity(claim, evidence):

    if not claim or not evidence:
        return 0.0

    claim_embedding = embedding_model.encode(
        claim, convert_to_numpy=True, normalize_embeddings=True
    )

    evidence_embedding = embedding_model.encode(
        evidence, convert_to_numpy=True, normalize_embeddings=True
    )

    score = np.dot(claim_embedding, evidence_embedding)

    return float(np.clip(score, 0.0, 1.0))


# ============================================================
# 6. NLI VERIFICATION
# ============================================================

def verify_with_nli(claim, evidence):

    if not claim or not evidence:
        return {"relation": "UNKNOWN", "score": 0.0}

    inputs = nli_tokenizer(
        evidence, claim, return_tensors="pt", truncation=True, max_length=512
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)[0]
    predicted_id = torch.argmax(probabilities).item()
    label = nli_model.config.id2label[predicted_id].lower()
    score = float(probabilities[predicted_id])

    if "entail" in label:
        relation = "SUPPORTED"
    elif "contrad" in label:
        relation = "CONTRADICTED"
    elif "neutral" in label:
        relation = "NEUTRAL"
    else:
        relation = "UNKNOWN"

    return {"relation": relation, "score": score}


# ============================================================
# 7. CONFIDENCE SCORE
# ============================================================

def calculate_confidence(semantic_score, nli_relation, nli_score):

    if nli_relation == "SUPPORTED":
        confidence = 0.60 * semantic_score + 0.40 * nli_score
    elif nli_relation == "CONTRADICTED":
        confidence = -(0.60 * semantic_score + 0.40 * nli_score)
    else:
        confidence = 0.50 * semantic_score

    return round(confidence, 4)


# ============================================================
# 8. HALLUCINATION CLASSIFICATION
# ============================================================

def classify_claim(confidence, nli_relation):

    if nli_relation == "SUPPORTED" and confidence >= 0.60:
        return "VERIFIED"

    if nli_relation == "CONTRADICTED":
        return "HALLUCINATED"

    if confidence < 0.20:
        return "UNCERTAIN"

    return "UNCERTAIN"


# ============================================================
# 9. VERIFY ONE CLAIM
# ============================================================

def verify_claim(claim):

    print(f"\nSearching evidence for:\n{claim}")

    evidence_list = retrieve_evidence(claim)

    if not evidence_list:
        return {
            "claim": claim,
            "status": "UNCERTAIN",
            "semantic_similarity": 0.0,
            "nli_relation": "NO_EVIDENCE",
            "nli_score": 0.0,
            "confidence": 0.0,
            "evidence": "",
            "source": "",
            "title": ""
        }

    best_result = None

    for item in evidence_list:
        evidence_text = item.get("content", "")

        if not evidence_text:
            continue

        similarity = semantic_similarity(claim, evidence_text)
        nli = verify_with_nli(claim, evidence_text)
        confidence = calculate_confidence(similarity, nli["relation"], nli["score"])
        status = classify_claim(confidence, nli["relation"])

        result = {
            "claim": claim,
            "status": status,
            "semantic_similarity": round(similarity, 4),
            "nli_relation": nli["relation"],
            "nli_score": round(nli["score"], 4),
            "confidence": confidence,
            "evidence": evidence_text,
            "source": item.get("url", ""),
            "title": item.get("title", "")
        }

        if best_result is None:
            best_result = result
        else:
            if abs(result["confidence"]) > abs(best_result["confidence"]):
                best_result = result

    if best_result is None:
        return {
            "claim": claim,
            "status": "UNCERTAIN",
            "semantic_similarity": 0.0,
            "nli_relation": "NO_EVIDENCE",
            "nli_score": 0.0,
            "confidence": 0.0,
            "evidence": "",
            "source": "",
            "title": ""
        }

    return best_result


# ============================================================
# 10. VERIFY ALL CLAIMS
# ============================================================

def verify_all_claims(claims):

    results = []

    if not claims:
        return results

    for i, claim in enumerate(claims, start=1):
        print("\n")
        print("=" * 65)
        print(f"CLAIM {i}")
        print("=" * 65)

        result = verify_claim(claim)
        results.append(result)

    return results


# ============================================================
# 11. FRESH SEARCH FOR HALLUCINATED CLAIM (WITH FALLBACK)
# ============================================================

def retrieve_correction_evidence(question, hallucinated_claim, max_results=5):

    print("\n")
    print("=" * 65)
    print("HALLUCINATION DETECTED")
    print("=" * 65)
    print("Hallucinated claim:")
    print(hallucinated_claim)
    print("\nSearching fresh evidence for correction...")

    if "tavily_client" not in globals() or tavily_client is None:
        print("Tavily client is not available.")
        return []

    # IMPORTANT:
    # Search the ORIGINAL QUESTION + BAD CLAIM.
    # This prevents unrelated results from dominating the correction.

    correction_query = (
        question + " Verify the correct factual answer. " + hallucinated_claim
    )

    try:
        # Attempt 1: trusted domains
        response = tavily_client.search(
            query=correction_query,
            search_depth="advanced",
            max_results=max_results,
            include_answer=False,
            include_domains=TRUSTED_DOMAINS
        )

        processed = _process_tavily_results(
            response.get("results", []), restrict_to_trusted=True
        )

        # Attempt 2: fallback to broader search
        if not processed:
            print("No trusted-domain correction evidence. Falling back...")
            response = tavily_client.search(
                query=correction_query,
                search_depth="advanced",
                max_results=max_results,
                include_answer=False
            )
            processed = _process_tavily_results(
                response.get("results", []), restrict_to_trusted=False
            )

        print("Fresh correction evidence found:", len(processed))

        return processed[:max_results]

    except Exception as e:
        print("Correction search error:", e)
        return []


# ============================================================
# 12. RAG-STYLE GROUNDED GENERATION
# ============================================================

def generate_rag_answer(question, max_new_tokens=150):
    """
    Retrieval-grounded generation: evidence is retrieved BEFORE
    generation, and Mistral is instructed to answer using only
    that evidence. This avoids relying on Mistral's parametric
    (and possibly outdated) internal knowledge for questions
    about current/changing facts.
    """

    evidence_list = retrieve_evidence(question, max_results=5)

    if not evidence_list:
        print("[RAG] No evidence found, falling back to parametric generation.")
        return generate_llm_answer(question, max_new_tokens)

    context_text = "\n\n".join(
        f"Source: {item.get('title', '')}\n{item.get('content', '')[:800]}"
        for item in evidence_list
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a factual assistant. Answer the question using "
                "ONLY the information in the provided context below. "
                "If the context doesn't contain the answer, say you're "
                "unsure rather than guessing. Do not use any outside "
                "knowledge, even if you think you know the answer."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context_text}\n\nQuestion: {question}"
        }
    ]

    return _generate_with_mistral(messages, max_new_tokens, max_input_length=3072)


# ============================================================
# 13. GENERATE CORRECTED / MITIGATED RESPONSE
# ============================================================

def generate_corrected_answer(question, original_answer, verification_results):

    if not verification_results:
        return "The answer could not be verified."

    # ========================================================
    # DETECT HALLUCINATED CLAIMS
    # ========================================================

    hallucinated_claims = [
        result for result in verification_results
        if result.get("status") == "HALLUCINATED"
    ]

    verified_claims = [
        result for result in verification_results
        if result.get("status") == "VERIFIED"
    ]

    uncertain_claims = [
        result for result in verification_results
        if result.get("status") == "UNCERTAIN"
    ]

    # ========================================================
    # FRESH CORRECTION EVIDENCE
    # ========================================================

    correction_evidence = []

    if hallucinated_claims:
        print("\n")
        print("=" * 75)
        print("MITIGATION / CORRECTION PHASE")
        print("=" * 75)
        print(f"Hallucinated claims detected: {len(hallucinated_claims)}")

        for bad_claim in hallucinated_claims:
            fresh_results = retrieve_correction_evidence(
                question,
                bad_claim.get("claim", ""),
                max_results=5
            )

            for item in fresh_results:
                correction_evidence.append(item)

    # ========================================================
    # BUILD EVIDENCE CONTEXT (with consistent Source IDs)
    # ========================================================

    url_to_source_id = {}
    evidence_context = ""

    def get_source_id(url):
        if url not in url_to_source_id:
            url_to_source_id[url] = f"S{len(url_to_source_id) + 1}"
        return url_to_source_id[url]

    for result in verified_claims:
        evidence_text = result.get("evidence", "")
        url = result.get("source", "")

        if len(evidence_text) > 1800:
            evidence_text = evidence_text[:1800] + "..."

        source_id = get_source_id(url)

        evidence_context += f"""

VERIFIED EVIDENCE {source_id}

Claim:
{result.get('claim', '')}

Evidence:
{evidence_text}

Source URL:
{url}

--------------------------------
"""

    for item in correction_evidence:
        evidence_text = item.get("content", "")
        url = item.get("url", "")

        if len(evidence_text) > 1800:
            evidence_text = evidence_text[:1800] + "..."

        source_id = get_source_id(url)

        evidence_context += f"""

FRESH CORRECTION EVIDENCE {source_id}

Evidence:
{evidence_text}

Source URL:
{url}

--------------------------------
"""

    # ========================================================
    # UNCERTAIN INFORMATION
    # ========================================================

    uncertain_text = ""
    for result in uncertain_claims:
        uncertain_text += "\n" + result.get("claim", "")

    # ========================================================
    # IF NO EVIDENCE
    # ========================================================

    if not evidence_context.strip():
        return (
            "The available evidence was insufficient "
            "to generate a verified answer."
        )

    # ========================================================
    # CORRECTION PROMPT
    # ========================================================

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict hallucination mitigation "
                "assistant. "
                "Your ONLY job is to answer the ORIGINAL "
                "QUESTION using the supplied verified "
                "evidence. "
                "Never use outside knowledge. "
                "Never guess. "
                "Never invent facts. "
                "Never invent or modify URLs. "
                "Never repeat a hallucinated claim unless "
                "fresh evidence explicitly proves it. "
                "Do not add unrelated information. "
                "If a fact is not supported by the supplied "
                "evidence, leave it out."
            )
        },
        {
            "role": "user",
            "content": f"""

ORIGINAL QUESTION:
{question}


ORIGINAL LLM ANSWER:
{original_answer}


VERIFICATION RESULT:

Verified claims:
{len(verified_claims)}

Hallucinated claims:
{len(hallucinated_claims)}

Uncertain claims:
{len(uncertain_claims)}


HALLUCINATED CLAIMS TO REMOVE:

{chr(10).join(r.get("claim", "") for r in hallucinated_claims)}


EVIDENCE AVAILABLE FOR THE CORRECTED ANSWER:

{evidence_context}


IMPORTANT RULES:

1. Answer the ORIGINAL QUESTION directly.
2. Keep factual information only when it is supported
   by the supplied evidence.
3. Remove every hallucinated claim unless the FRESH
   CORRECTION EVIDENCE clearly proves the corrected fact.
4. Do NOT repeat a hallucinated claim.
5. Do NOT add extra facts just because you know them.
6. Do NOT mention unrelated topics.
7. Do NOT mention the verification process in the final
   answer.
8. Do NOT write fake URLs.
9. Do NOT change any URL.
10. For a source citation, use EXACTLY this format:
   [SOURCE:S1]
   [SOURCE:S2]
   etc.
11. Only use SOURCE IDs that actually appear in the
    supplied evidence.
12. Keep the final answer short and natural.
13. Do not write [Source] without a SOURCE ID.

FINAL ANSWER ONLY:
"""
        }
    ]

    corrected_answer = _generate_with_mistral(
        messages,
        max_new_tokens=180,
        max_input_length=4096
    )

    # ========================================================
    # REPLACE SOURCE IDs WITH REAL VERIFIED URLs
    # ========================================================

    for url, source_id in url_to_source_id.items():
        corrected_answer = corrected_answer.replace(
            f"[SOURCE:{source_id}]",
            f"[Source: {url}]"
        )

    corrected_answer = re.sub(
        r"\[SOURCE:S\d+\]", "", corrected_answer, flags=re.IGNORECASE
    )

    corrected_answer = corrected_answer.replace("[Source]", "")

    corrected_answer = corrected_answer.strip()

    if not corrected_answer:
        return (
            "A corrected answer could not be generated "
            "from the verified evidence."
        )

    return corrected_answer


# ============================================================
# CELL 4 COMPLETE
# ============================================================

print("=" * 75)
print("CELL 4 FUNCTIONS LOADED SUCCESSFULLY")
print("=" * 75)

CELL 4 FUNCTIONS LOADED SUCCESSFULLY


In [ ]:
# ============================================================
# CELL 5: COMPLETE HALLUCINATION DETECTION PIPELINE
# ============================================================

def run_hallucination_system(question):

    print("\n")
    print("=" * 75)
    print("NLP HALLUCINATION DETECTION SYSTEM")
    print("=" * 75)

    # ========================================================
    # STEP 1: LLM GENERATION
    # ========================================================
    #
    # CHANGE: if the question itself is time-sensitive (asks about
    # something current/recent — e.g. "who is the current president",
    # "who won the recent world cup"), we use RAG-grounded generation
    # instead of pure parametric generation. This retrieves evidence
    # BEFORE generating, so Mistral doesn't have to rely on its own
    # (possibly outdated) internal knowledge for facts that change
    # over time.

    print("\n[1/7] Generating answer using Mistral...")

    if is_time_sensitive(question):
        print("[Time-sensitive question detected] Using RAG-grounded generation...")
        original_answer = generate_rag_answer(question)
    else:
        original_answer = generate_llm_answer(question)

    print("\nLLM ANSWER:")
    print("-" * 60)
    print(original_answer)

    # ========================================================
    # EMPTY ANSWER CHECK
    # ========================================================

    if not original_answer.strip():
        print("\nERROR: Mistral generated an empty answer.")
        return {
            "question": question,
            "original_answer": "",
            "claims": [],
            "corrected_answer": "Unable to generate an answer."
        }

    # ========================================================
    # STEP 2: CLAIM EXTRACTION
    # ========================================================

    print("\n[2/7] Extracting claims...")

    claims = extract_claims(original_answer)

    print(f"Found {len(claims)} claims.")

    for i, claim in enumerate(claims, start=1):
        print(f"\nClaim {i}: {claim}")

    # ========================================================
    # NO CLAIMS CHECK
    # ========================================================

    if not claims:
        print("\nNo factual claims were extracted.")
        return {
            "question": question,
            "original_answer": original_answer,
            "claims": [],
            "corrected_answer": original_answer
        }

    # ========================================================
    # STEP 3-6: EVIDENCE RETRIEVAL + VERIFICATION
    # ========================================================

    print("\n[3/7] Retrieving external evidence...")
    print("[4/7] Calculating semantic similarity...")
    print("[5/7] Running NLI...")
    print("[6/7] Calculating confidence + detecting hallucination...")

    results = verify_all_claims(claims)

    # ========================================================
    # CLAIM VERIFICATION RESULTS
    # ========================================================

    print("\n")
    print("=" * 75)
    print("CLAIM VERIFICATION RESULTS")
    print("=" * 75)

    for i, result in enumerate(results, start=1):
        print(f"\nCLAIM {i}:")
        print(result.get("claim", ""))
        print("\nStatus:", result.get("status", "UNCERTAIN"))
        print("Semantic Similarity:", result.get("semantic_similarity", 0.0))
        print("NLI:", result.get("nli_relation", "UNKNOWN"))
        print("NLI Score:", result.get("nli_score", 0.0))
        print("Confidence:", result.get("confidence", 0.0))
        print("Source:", result.get("source", ""))

    # ========================================================
    # COUNT RESULTS BEFORE CORRECTION
    # ========================================================

    verified = sum(r.get("status") == "VERIFIED" for r in results)
    hallucinated = sum(r.get("status") == "HALLUCINATED" for r in results)
    uncertain = sum(r.get("status") == "UNCERTAIN" for r in results)

    # ========================================================
    # HALLUCINATION STATUS
    # ========================================================

    print("\n")
    print("=" * 75)
    print("HALLUCINATION STATUS")
    print("=" * 75)

    if hallucinated > 0:
        print(f"Hallucination detected: {hallucinated} claim(s)")
        for i, result in enumerate(results, start=1):
            if result.get("status") == "HALLUCINATED":
                print(f"\nHallucinated Claim {i}:")
                print(result.get("claim", ""))
    else:
        print("No hallucinated claims detected.")

    # ========================================================
    # STEP 7: CORRECTION / MITIGATION
    # ========================================================

    print("\n[7/7] Generating corrected response...")

    corrected_answer = generate_corrected_answer(question, original_answer, results)

    # ========================================================
    # FINAL CORRECTED RESPONSE
    # ========================================================

    print("\n")
    print("=" * 75)
    print("       CORRECTED VERIFIED RESPONSE")
    print("=" * 75)
    print(corrected_answer)

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n")
    print("=" * 75)
    print("SUMMARY")
    print("=" * 75)
    print("Total Claims:", len(results))
    print("Verified:", verified)
    print("Hallucinated:", hallucinated)
    print("Uncertain:", uncertain)

    # ========================================================
    # RETURN EVERYTHING
    # ========================================================

    return {
        "question": question,
        "original_answer": original_answer,
        "claims": results,
        "corrected_answer": corrected_answer,
        "verified_count": verified,
        "hallucinated_count": hallucinated,
        "uncertain_count": uncertain
    }


# ============================================================
# CELL 5 READY
# ============================================================

print("=" * 75)
print("CELL 5 PIPELINE LOADED SUCCESSFULLY")
print("=" * 75)

CELL 5 PIPELINE LOADED SUCCESSFULLY


In [ ]:
# ============================================================
# CELL 6: INTERACTIVE TEST
# ============================================================

print("=" * 75)
print("INTERACTIVE HALLUCINATION DETECTION TEST")
print("=" * 75)

while True:

    question = input(
        "\nEnter your question "
        "(type 'exit' to stop): "
    )


    # --------------------------------------------------------
    # EXIT
    # --------------------------------------------------------

    if question.lower().strip() == "exit":

        print("\nSystem stopped.")

        break


    # --------------------------------------------------------
    # EMPTY QUESTION
    # --------------------------------------------------------

    if not question.strip():

        print(
            "Please enter a question."
        )

        continue


    # --------------------------------------------------------
    # RUN HALLUCINATION SYSTEM
    # --------------------------------------------------------

    try:

        result = run_hallucination_system(
            question
        )


    except KeyboardInterrupt:

        print(
            "\n\nSystem interrupted by user."
        )

        break


    except Exception as e:

        print("\n")
        print("=" * 75)
        print("ERROR")
        print("=" * 75)

        print(
            type(e).__name__ + ":",
            str(e)
        )

        print("=" * 75)

        print(
            "\nPlease check the error above."
        )

INTERACTIVE HALLUCINATION DETECTION TEST

Enter your question (type 'exit' to stop): who is the current president of india?


NLP HALLUCINATION DETECTION SYSTEM

[1/7] Generating answer using Mistral...
[Time-sensitive question detected] Using RAG-grounded generation...
[Time-sensitive claim detected]
Searching trusted domains...
Usable evidence: 5

LLM ANSWER:
------------------------------------------------------------
The current president of India is Droupadi Murmu, who took office on 25 July 2022.

[2/7] Extracting claims...
Found 1 claims.

Claim 1: The current president of India is Droupadi Murmu, who took office on 25 July 2022.

[3/7] Retrieving external evidence...
[4/7] Calculating semantic similarity...
[5/7] Running NLI...
[6/7] Calculating confidence + detecting hallucination...


CLAIM 1

Searching evidence for:
The current president of India is Droupadi Murmu, who took office on 25 July 2022.
[Time-sensitive claim detected]
Searching trusted domains...
Usable evidence: 